# Freight Rate Prediction — Exploratory Data Analysis

## 1. Load the data
Three files: `train_test.csv` (labeled history), `validation.csv` (12,000 loads to predict),
and `december_chart_inputs.csv` (one fixed load on every day of December 2025).

In [ ]:
import pandas as pd

# Load each file into a table (DataFrame)
train = pd.read_csv("../data/train_test.csv")
val = pd.read_csv("../data/validation.csv")
dec = pd.read_csv("../data/december_chart_inputs.csv")

# How big is each table? (rows, columns)
print("train:", train.shape)
print("validation:", val.shape)
print("december:", dec.shape)

train.head()

train: (48000, 14)
validation: (12000, 13)
december: (31, 7)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


## 2. Missing values and basic statistics

In [2]:
# Column types and non-missing counts
train.info()

# How many missing values in each column
print(train.isna().sum())

# Summary statistics for numeric columns (min, max, mean...)
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance   

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


**Findings**
- `weight` has 300 missing values and `market_index` has 374 (both under 1%). All other columns are complete.
- `weight` has a minimum of -47,500 lb, which is impossible, so some values are wrong.
- `posted_rate` ranges from $57 to $25,533, while the middle half lies between about $1,250 and $3,330,
  so there are likely extreme price errors.
- `date` is stored as text and must be converted to a real date.

## 3. Comparing the three files

In [7]:
# Date range of each file
for name, df in [("train", train), ("validation", val)]:
    dates = pd.to_datetime(df["date"])
    print(name, dates.min().date(), "to", dates.max().date())

# Which columns exist in train but not in the December file?
print("Missing from December:", set(train.columns) - set(dec.columns))

# How common is each equipment type?
train["equipment"].value_counts()

train 2025-01-01 to 2025-10-31
validation 2025-11-01 to 2025-12-31
Missing from December: {'rate_per_mile', 'quote_signal', 'quote_ratio', 'delivery_lat', 'posted_rate', 'pickup_lon', 'pickup_lat', 'market_index', 'load_id', 'delivery_lon'}


equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

**Findings**
- Training data covers Jan 1 – Oct 31, 2025, and validation covers Nov 1 – Dec 31, 2025.
  Validation is entirely in the future, so the model must be tested the same way:
  train on earlier months, test on the latest months (time-based split).
- The December file lacks `market_index`, `quote_signal` and the coordinates.
  The model will therefore use only the columns available in all files:
  pickup, delivery, distance, equipment, weight and date.
- Equipment: Dry Van 57%, Reefer 25%, Flatbed 18%. No inconsistent spellings.
  The December load is a Dry Van, the most common type.